# 06_02 · Modelo Final — Capa 2 (confirmación mecánica)

Construye los dos artefactos de la Capa 2:

| Artefacto | Datos | Papel |
|---|---|---|
| `models/kit_model.pkl` | KIT Karlsruhe (33 exp., 280 features, 500 Hz) | **Prototipo** — valida el concepto con sensores externos |
| `models/cnc_model.pkl` | CNC Mill (18 exp., 183 features, señales internas) | **Cold-start** de planta — reentrenable con datos reales |

Las configuraciones vienen de los notebooks de experimentación
(`05_03_Modelos_Industrial`, `05_02_Modelos_CNC`); aquí solo se construye, valida
con LOO y guarda. Si la experimentación encuentra una configuración mejor, se
actualiza aquí y se re-ejecuta.


In [1]:
import sys; sys.path.insert(0, '../../src')
import warnings; warnings.filterwarnings('ignore')
import os, pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (f1_score, recall_score, precision_score,
                              roc_auc_score, classification_report)

from paths import PROCESSED, MODELS
os.makedirs(MODELS, exist_ok=True)
loo = LeaveOneOut()

def loo_metrics(pipe, X, y):
    """Métricas LOO completas (las que consumen 06_04 y 07)."""
    yp = cross_val_predict(pipe, X, y, cv=loo)
    pr = cross_val_predict(pipe, X, y, cv=loo, method='predict_proba')[:, 1]
    return {
        'f1':        round(float(f1_score(y, yp, zero_division=0)), 3),
        'recall':    round(float(recall_score(y, yp, zero_division=0)), 3),
        'precision': round(float(precision_score(y, yp, zero_division=0)), 3),
        'auc':       round(float(roc_auc_score(y, pr)), 3),
    }


## 1. Prototipo KIT — `kit_model.pkl`

Configuración validada en `03_03`/`04_03`: Random Forest con selección de features.


In [2]:
kit_df = pd.read_csv(os.path.join(PROCESSED, 'industrial_features_v2.csv'))
feat_kit = [c for c in kit_df.columns
            if c not in ('trial', 'fault_type', 'fault_name', 'anomaly', 'n_samples')]
X_kit = kit_df[feat_kit].fillna(0).values
y_kit = kit_df['anomaly'].values

# Configuración elegida en la experimentación (03_03 / 05_03)
KIT_PARAMS = dict(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1)
kit_pipe = Pipeline([
    ('var', VarianceThreshold(threshold=0.0)),
    ('sel', SelectKBest(f_classif, k=30)),
    ('sc',  StandardScaler()),
    ('clf', RandomForestClassifier(**KIT_PARAMS)),
])

print(f'KIT: {len(kit_df)} experimentos | {len(feat_kit)} features')
metrics_kit = loo_metrics(kit_pipe, X_kit, y_kit)
print('Métricas LOO:', metrics_kit)

kit_pipe.fit(X_kit, y_kit)
kit_bundle = {
    'pipeline':      kit_pipe,
    'feature_names': feat_kit,
    'params':        KIT_PARAMS,
    'metrics_loo':   metrics_kit,
    'n_experiments': len(kit_df),
}
with open(os.path.join(MODELS, 'kit_model.pkl'), 'wb') as f:
    pickle.dump(kit_bundle, f)
print('Guardado: models/kit_model.pkl')


KIT: 33 experimentos | 224 features
Métricas LOO: {'f1': 0.667, 'recall': 0.667, 'precision': 0.667, 'auc': 0.707}
Guardado: models/kit_model.pkl


## 2. Cold-start CNC — `cnc_model.pkl`

Configuración elegida en `05_02_Modelos_CNC`: GradientBoosting con stumps
(`max_depth=1`) — la única familia que superó el baseline trivial (F1=0.714).


In [3]:
cnc_df = pd.read_csv(os.path.join(PROCESSED, 'cnc_aggregated.csv'))
meta = ['experiment', 'material', 'tool_condition', 'target']
feat_cnc = [c for c in cnc_df.columns if c not in meta]
X_cnc = cnc_df[feat_cnc].fillna(0).values
y_cnc = cnc_df['target'].values

# Configuración elegida en 05_02 (grid de 96 configs, LOO)
CNC_PARAMS = dict(n_estimators=200, learning_rate=0.2, max_depth=1,
                  subsample=1.0, random_state=42)
cnc_pipe = Pipeline([
    ('var', VarianceThreshold(threshold=0.0)),
    ('sc',  StandardScaler()),
    ('clf', GradientBoostingClassifier(**CNC_PARAMS)),
])

f1_trivial = f1_score(y_cnc, np.ones_like(y_cnc))
print(f'CNC: {len(cnc_df)} experimentos | {len(feat_cnc)} features | trivial F1={f1_trivial:.3f}')
metrics_cnc = loo_metrics(cnc_pipe, X_cnc, y_cnc)
print('Métricas LOO:', metrics_cnc)

cnc_pipe.fit(X_cnc, y_cnc)
cnc_bundle = {
    'pipeline':      cnc_pipe,
    'feature_names': feat_cnc,
    'best_params':   CNC_PARAMS,
    'metrics_loo':   metrics_cnc,
    'f1_trivial':    round(float(f1_trivial), 3),
    'n_experiments': len(cnc_df),
}
with open(os.path.join(MODELS, 'cnc_model.pkl'), 'wb') as f:
    pickle.dump(cnc_bundle, f)
print('Guardado: models/cnc_model.pkl')


CNC: 18 experimentos | 183 features | trivial F1=0.714
Métricas LOO: {'f1': 0.909, 'recall': 1.0, 'precision': 0.833, 'auc': 0.85}
Guardado: models/cnc_model.pkl


## 3. Resumen Capa 2

Ambos bundles incluyen el `Pipeline` completo (selección + escalado + clasificador),
los nombres de las features esperadas y las métricas LOO — autocontenidos para que
`07_Sistema_Industrial` y `06_04_Evaluacion_Final` los consuman sin reentrenar.

**Reentrenamiento en planta (Capa 2 CNC):** con cada experimento etiquetado nuevo,
`pipeline.fit(X_acumulado, y_acumulado)` y volver a guardar el bundle.
